In [ ]:
# Step 2: Preparing the Dataset

import pandas as pd
from datasets import load_dataset
import os
from PIL import Image
import io

# 1. Download the fashion product dataset from HuggingFace
print("Loading dataset from HuggingFace...")
dataset = load_dataset("ashraq/fashion-product-images-small", split="train")

# 2. Convert to pandas DataFrame for easier manipulation
df = dataset.to_pandas()

# 3. Display basic information about the dataset
print(f"\nDataset loaded successfully!")
print(f"Total products: {len(df)}")
print(f"\nColumns available: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# 4. Create a data structure for products
class Product:
    """Data structure to hold product information"""
    def __init__(self, product_id, name, price, category, image, description=None, features=None):
        self.id = product_id
        self.name = name
        self.price = price
        self.category = category
        self.image = image  # PIL Image object or image bytes
        self.description = description
        self.features = features if features else []
    
    def to_dict(self):
        """Convert product to dictionary format"""
        return {
            'id': self.id,
            'name': self.name,
            'price': self.price,
            'category': self.category,
            'image': self.image,
            'description': self.description,
            'features': self.features
        }

# 5. Create a list to store all product objects
products = []

# 6. Process and organize product data
print("\nProcessing products...")

# Sample a subset for demonstration (first 100 products)
sample_size = min(100, len(df))
df_sample = df.head(sample_size)

for idx, row in df_sample.iterrows():
    try:
        # Extract product information from dataset
        product_id = row.get('id', f"PROD_{idx}")
        name = row.get('productDisplayName', f"Product {idx}")
        
        # For price, create a realistic price based on category or use default
        # The dataset doesn't have price, so we'll generate reasonable prices
        category = row.get('subCategory', 'Unknown')
        price = round(10 + (hash(str(idx)) % 500) + 0.99, 2)  # Random price between $10-$510
        
        # Get image
        image = row.get('image')
        if image is not None:
            # If image is stored as bytes, convert to PIL Image
            if isinstance(image, bytes):
                image_obj = Image.open(io.BytesIO(image))
            else:
                image_obj = image
        else:
            image_obj = None
        
        # Create product object
        product = Product(
            product_id=product_id,
            name=name,
            price=price,
            category=category,
            image=image_obj,
            description=None,  # Will be generated later
            features=[]
        )
        
        products.append(product)
        
    except Exception as e:
        print(f"Error processing product at index {idx}: {e}")
        continue

# 7. Verify data access
print(f"\n✅ Successfully loaded {len(products)} products")
print("\nSample product information:")
sample_product = products[0]
print(f"Product ID: {sample_product.id}")
print(f"Name: {sample_product.name}")
print(f"Price: ${sample_product.price}")
print(f"Category: {sample_product.category}")
print(f"Has Image: {sample_product.image is not None}")

if sample_product.image is not None:
    print(f"Image type: {type(sample_product.image)}")
    print(f"Image size: {sample_product.image.size if hasattr(sample_product.image, 'size') else 'N/A'}")

# 8. Create a DataFrame summary for easy viewing
product_summary = pd.DataFrame([{
    'id': p.id,
    'name': p.name[:50] + '...' if len(p.name) > 50 else p.name,
    'price': p.price,
    'category': p.category,
    'has_image': p.image is not None
} for p in products])

print("\nProduct Summary:")
print(product_summary.head(10))

# 9. Checkpoint verification
print("\n" + "="*50)
print("CHECKPOINT: Dataset Ready")
print("="*50)
print(f"✅ Total products loaded: {len(products)}")
print(f"✅ Products with images: {sum(1 for p in products if p.image is not None)}")
print(f"✅ Sample categories: {df['subCategory'].unique()[:5].tolist()}")
print("\n✅ Dataset verification complete! Ready for next step.")
print("="*50)